# FDA Week 04 Instructor Lab v2
## 거래량 기반 기술적 분석, 통합 투자전략과 위험관리

이번 버전의 핵심 수정사항:

1. 거래정지일을 indicator/execution clock에서 제외
2. IS/OOS 거래횟수를 각 기간에서 다시 계산
3. IS/OOS exposure-matched B&H를 각각 계산
4. Toy state-machine 자동 PASS
5. +5 거래일 Volume Event Study + Newey-West t-stat


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from FDA_week04_reference_v21 import (
    prepare_single_stock,
    build_indicators,
    correlation_inputs,
    public_indicator_sanity,
)

from FDA_week04_instructor_solution_v21 import (
    run_long_only_backtest,
    buy_and_hold_returns,
    summarize_strategy,
    exposure_matched_bh_period,
    split_masks_by_last_two_years,
    period_strategy_summary,
    run_toy_state_machine,
    build_volume_event_study,
)

DATA_FILE = Path("FDA_kospi200_daily.parquet")

df_all = pd.read_parquet(DATA_FILE)
df_all["Date"] = pd.to_datetime(df_all["Date"])
df_all["Symbol"] = (
    df_all["Symbol"]
    .astype("string")
    .str.zfill(6)
)

print("shape:", df_all.shape)
print("date range:", df_all["Date"].min().date(), "->", df_all["Date"].max().date())
print("unique symbols:", df_all["Symbol"].nunique())


## 1. 거래정지일과 Trading-day clock

원자료의 거래정지 행은 보존하지만, 지표와 체결은 실제 거래관측치만 사용합니다.

삼성전자 2018년 액면분할 거래정지 구간을 확인합니다.


In [ ]:
raw_stock = prepare_single_stock(
    df_all,
    "005930",
    trading_only=False,
)

suspension = raw_stock.loc[
    raw_stock["Date"].between("2018-04-20", "2018-05-15"),
    ["Date","Close","AdjustedClose","Volume","TradingDay"]
]

display(suspension)

print("raw rows =", len(raw_stock))
print("non-trading rows =", int((~raw_stock["TradingDay"]).sum()))


In [ ]:
stock = prepare_single_stock(
    df_all,
    "005930",
    trading_only=True,
)

stock = build_indicators(stock)

print(public_indicator_sanity(stock))
print("trading-day rows =", len(stock))


## 2. Corporate-action adjustment

$$
f_t=\frac{AdjustedClose_t}{Close_t}
$$

$$
AdjOpen=Open\times f,\quad
AdjHigh=High\times f,\quad
AdjLow=Low\times f,\quad
AdjVolume=Volume/f
$$


In [ ]:
window = stock.loc[
    stock["Date"].between("2018-04-20", "2018-05-15"),
    [
        "Date","Close","AdjustedClose",
        "Volume","AdjVolume","AdjustmentFactor_Lab"
    ]
]

display(window)


In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(window["Date"], window["Close"], label="Raw Close")
ax.plot(window["Date"], window["AdjustedClose"], label="Adjusted Close")
ax.set_title("Samsung Electronics: raw vs adjusted close")
ax.legend()
plt.show()


## 3. 거래량 지표

In [ ]:
view = stock.dropna(
    subset=["VolumeRatio20","TradingValueRatio20"]
).tail(500)

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(view["Date"], view["VolumeRatio20"], label="VolumeRatio20")
ax.plot(view["Date"], view["TradingValueRatio20"], label="TradingValueRatio20")
ax.axhline(1.5, linestyle="--", label="1.5")
ax.set_title("Volume ratio vs trading-value ratio")
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(view["Date"], view["OBV"])
ax.set_title("On-Balance Volume")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(view["Date"], view["RSI14"], label="RSI14")
ax.plot(view["Date"], view["MFI14"], label="MFI14")
ax.axhline(70, linestyle="--")
ax.axhline(30, linestyle="--")
ax.set_title("RSI vs MFI")
ax.legend()
plt.show()


## 4. Indicator redundancy

In [ ]:
corr = correlation_inputs(stock).dropna().corr()
display(corr.round(3))

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Indicator correlation matrix")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 5. Composite signals

In [ ]:
display(
    stock[
        [
            "Date","AdjustedClose",
            "TrendSignal","MomentumSignal","VolumeSignal",
            "Score","RSI14","HighVol",
            "EntryCondition_COMP",
            "EntryCondition_COMP_RM",
            "ExitCondition",
        ]
    ].dropna().tail(20)
)


## 6. Toy state-machine validation

실제 주식 백테스트 전에 stop, reset, re-entry가 규칙대로 움직이는지 검증합니다.


In [ ]:
toy = pd.read_csv("FDA_week04_toy_backtest_input_v21.csv")
expected = pd.read_csv("FDA_week04_toy_backtest_expected_v21.csv")

actual = run_toy_state_machine(toy)

def normalize_toy_frame(x):
    y = x.copy()

    y["Day"] = pd.to_numeric(y["Day"], errors="raise").astype(int)

    y["ExpectedStateAfterClose"] = (
        y["ExpectedStateAfterClose"]
        .fillna("")
        .astype(str)
    )

    # Blank or numeric entry day -> canonical integer string.
    entry_num = pd.to_numeric(
        y["ExpectedEntryExecutionDay"],
        errors="coerce",
    )
    y["ExpectedEntryExecutionDay"] = [
        "" if pd.isna(v) else str(int(v))
        for v in entry_num
    ]

    y["ExpectedStopType"] = (
        y["ExpectedStopType"]
        .fillna("")
        .astype(str)
    )

    y["ExpectedResetSatisfied"] = (
        pd.to_numeric(
            y["ExpectedResetSatisfied"],
            errors="raise",
        )
        .astype(int)
    )

    return y

toy_pass = normalize_toy_frame(actual).equals(
    normalize_toy_frame(expected)
)

display(actual)

print(
    "FDA_WEEK04_TOY_STATE_MACHINE_PASS"
    if toy_pass
    else "FDA_WEEK04_TOY_STATE_MACHINE_FAIL"
)

assert toy_pass


## 7. COMP / COMP_RM 백테스트

In [ ]:
BASELINE_ROUND_TRIP_COST = 0.0025  # 수업 전 최신 가정 확인

comp, comp_info = run_long_only_backtest(
    stock,
    mode="COMP",
    round_trip_cost=BASELINE_ROUND_TRIP_COST,
    burn_in=120,
)

comp_rm, comp_rm_info = run_long_only_backtest(
    stock,
    mode="COMP_RM",
    round_trip_cost=BASELINE_ROUND_TRIP_COST,
    burn_in=120,
)

print(comp_info)
print(comp_rm_info)


In [ ]:
events = comp_rm.loc[
    comp_rm["Event"].ne(""),
    [
        "Date","Event","ExecutionPrice",
        "EntryPrice","StopPrice",
        "StrategyReturn","State",
        "EntryExecuted",
    ]
]

display(events.head(30))


## 8. 전체기간 성과 비교

In [ ]:
bh = buy_and_hold_returns(stock, burn_in=120)

evaluation_mask = pd.Series(
    np.arange(len(stock)) >= 120,
    index=stock.index,
)

def whole_summary():
    rows = []

    bh_eval = bh.loc[evaluation_mask].reset_index(drop=True)
    w_bh = pd.Series(1.0, index=bh_eval.index)

    s = summarize_strategy(
        bh_eval,
        w_bh,
        n_trades=1,
        initial_weight=0.0,
    )
    s["strategy"] = "BH"
    rows.append(s)

    for name, bt in [
        ("COMP", comp),
        ("COMP_RM", comp_rm),
    ]:
        part = bt.loc[evaluation_mask].copy()

        s = summarize_strategy(
            part["StrategyReturn"].reset_index(drop=True),
            part["WeightClose"].reset_index(drop=True),
            n_trades=int(part["EntryExecuted"].sum()),
            initial_weight=0.0,
        )
        s["strategy"] = name
        rows.append(s)

    # Exposure-matched B&H using COMP exposure.
    comp_part = comp.loc[evaluation_mask].copy()
    embh_r, avg_w = exposure_matched_bh_period(
        bh_eval,
        comp_part["WeightClose"].reset_index(drop=True),
    )

    s = summarize_strategy(
        embh_r,
        pd.Series(avg_w, index=embh_r.index),
        n_trades=1,
        initial_weight=0.0,
    )
    s["strategy"] = "BH_EXPOSURE_MATCHED_COMP"
    rows.insert(1, s)

    return pd.DataFrame(rows).set_index("strategy")

perf = whole_summary()
display(perf.round(4))


In [ ]:
wealth = pd.DataFrame({
    "Date": stock["Date"],
    "BH": (1 + bh).cumprod(),
    "COMP": (1 + comp["StrategyReturn"]).cumprod(),
    "COMP_RM": (1 + comp_rm["StrategyReturn"]).cumprod(),
})

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(wealth["Date"], wealth["BH"], label="Buy & Hold")
ax.plot(wealth["Date"], wealth["COMP"], label="COMP")
ax.plot(wealth["Date"], wealth["COMP_RM"], label="COMP_RM")
ax.set_yscale("log")
ax.set_title("Cumulative wealth")
ax.legend()
plt.show()


In [ ]:
def drawdown_curve(r):
    w = (1 + pd.Series(r).fillna(0.0)).cumprod()
    return w / w.cummax() - 1.0

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(stock["Date"], drawdown_curve(bh), label="BH")
ax.plot(stock["Date"], drawdown_curve(comp["StrategyReturn"]), label="COMP")
ax.plot(stock["Date"], drawdown_curve(comp_rm["StrategyReturn"]), label="COMP_RM")
ax.set_title("Drawdown")
ax.legend()
plt.show()


## 9. 거래비용 민감도

In [ ]:
cost_rows = []

for multiple in [0, 1, 2]:
    c = BASELINE_ROUND_TRIP_COST * multiple

    bt, info = run_long_only_backtest(
        stock,
        mode="COMP_RM",
        round_trip_cost=c,
        burn_in=120,
    )

    part = bt.loc[evaluation_mask].copy()

    s = summarize_strategy(
        part["StrategyReturn"].reset_index(drop=True),
        part["WeightClose"].reset_index(drop=True),
        n_trades=int(part["EntryExecuted"].sum()),
        initial_weight=0.0,
    )

    s["cost_multiple"] = multiple
    s["round_trip_cost"] = c
    cost_rows.append(s)

cost_table = (
    pd.DataFrame(cost_rows)
    .set_index("cost_multiple")
)

display(cost_table.round(4))


## 10. IS / OOS + 기간별 Exposure-matched B&H

마지막 2년을 OOS로 사용합니다.  
각 구간에서 평균 exposure를 새로 계산하므로 미래 exposure 정보를 사용하지 않습니다.


### Turnover 경계 처리

IS/OOS 하위기간의 첫날 이미 보유하고 있던 포지션은 그 하위기간에서 새로 매수한 것이 아닙니다.
따라서 subperiod turnover 계산에서는 첫 관측치의 기존 비중을 신규 turnover로 계산하지 않습니다.


In [ ]:
is_mask, oos_mask, cutoff = split_masks_by_last_two_years(stock)
is_mask &= evaluation_mask
oos_mask &= evaluation_mask

def period_table(mask, label):
    rows = []

    bh_part = bh.loc[mask].reset_index(drop=True)

    # BH
    s = summarize_strategy(
        bh_part,
        pd.Series(1.0, index=bh_part.index),
        n_trades=1,
        initial_weight=None,
    )
    s["period"] = label
    s["strategy"] = "BH"
    rows.append(s)

    # COMP / COMP_RM
    for name, bt in [
        ("COMP", comp),
        ("COMP_RM", comp_rm),
    ]:
        part = bt.loc[mask].copy()

        s = summarize_strategy(
            part["StrategyReturn"].reset_index(drop=True),
            part["WeightClose"].reset_index(drop=True),
            n_trades=int(part["EntryExecuted"].sum()),
            initial_weight=None,
        )
        s["period"] = label
        s["strategy"] = name
        rows.append(s)

    # Exposure-matched B&H for COMP and COMP_RM separately.
    for name, bt in [
        ("BH_EM_COMP", comp),
        ("BH_EM_COMP_RM", comp_rm),
    ]:
        weights = bt.loc[mask, "WeightClose"].reset_index(drop=True)

        em_r, avg_w = exposure_matched_bh_period(
            bh_part,
            weights,
        )

        s = summarize_strategy(
            em_r,
            pd.Series(avg_w, index=em_r.index),
            n_trades=1,
            initial_weight=None,
        )
        s["period"] = label
        s["strategy"] = name
        rows.append(s)

    return rows

period_perf = pd.DataFrame(
    period_table(is_mask, "IS")
    + period_table(oos_mask, "OOS")
)

print("OOS cutoff:", cutoff.date())
display(
    period_perf
    .set_index(["period","strategy"])
    .round(4)
)


## 11. +5 거래일 Volume Event Study

- VolumeRatio20 > 1.5
- 직전 20 거래관측치 내 이벤트가 없는 첫 날
- UP / DOWN 분리
- KOSPI200 PIT 동일가중 시장수익률 차감
- event-date calendar-time 평균
- Newey-West(HAC, lag=4) t-stat


In [ ]:
event_rows, event_summary = build_volume_event_study(
    df_all,
    horizon=5,
    event_threshold=1.5,
    exclusion_window=20,
)

display(event_summary.round(4))

print("event rows:", len(event_rows))
display(
    event_rows[
        [
            "Date","Symbol","Name",
            "VolumeRatio20_ES",
            "Direction","StockFwd5",
            "MktFwd5","CAR5"
        ]
    ].head(20)
)


## 12. 핵심 해석

1. 거래정지일을 제거하면 20일 지표는 실제 20 거래관측치를 의미한다.
2. 거래량 지표는 가격 지표와 낮은 상관을 보일 수 있어 비중복 정보를 제공할 가능성이 있다.
3. 기술적 신호가 gross 기준으로 작동해도 높은 turnover와 비용 때문에 net 성과가 사라질 수 있다.
4. RM 전략은 수익률보다 MDD와 변동성을 줄이는 역할로 평가해야 한다.
5. OOS 2년 성과 차이는 표본오차가 크므로 과도하게 일반화하지 않는다.
